# Clinical AI Tool Agent — Demo

**EDUCATIONAL USE ONLY.** This is a teaching notebook. The agent's outputs are not medical advice.

Three runs demonstrate the agent's three primary code paths:
1. A pure-knowledge question (retriever → final answer, no tools).
2. A computation question (retriever → tool call → final answer).
3. A combined question (retriever → tool → re-prompt → final answer).

**Prerequisites:**
1. `ollama pull tinyllama`
2. `pip install -e .[dev]` from the repo root.

> A clean, library-using demo of the packaged code. For the original full walkthrough this project was extracted from — including all model outputs and the exploratory iteration — see [`walkthrough.ipynb`](walkthrough.ipynb).


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

from clinical_ai_tool_agent.agent import build_default_llm, run_pipeline_react
from clinical_ai_tool_agent.knowledge import build_retriever
from clinical_ai_tool_agent.tools import build_tools

llm = build_default_llm()
retriever = build_retriever()
tools = build_tools()
print(f'Loaded {len(tools)} tools:', [t.name for t in tools])

c:\Users\siegk\OneDrive\Documents\GenAI-Portfolio\clinical-ai-tool-agent\src\clinical_ai_tool_agent\agent.py:120: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  return ChatOllama(


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'


Loaded 3 tools: ['calc_bmi', 'calc_map', 'calc_anion_gap']


## Path 1 — knowledge retrieval only

In [2]:
trace = run_pipeline_react(
    'What are the red flags for fever and cough? Education only.',
    llm=llm, retriever=retriever, tools=tools,
)
print('Tool calls:', trace.tool_calls)
print(f'\n--- Final ({trace.steps} step(s), {trace.latency_s}s) ---')
print(trace.final_answer)

Tool calls: []

--- Final (5 step(s), 40.17s) ---
None


## Path 2 — computation via tool calling

In [3]:
trace = run_pipeline_react(
    'Given labs: Na 138, Cl 100, HCO3 22. Compute anion gap and outline interpretation.',
    llm=llm, retriever=retriever, tools=tools,
)
print('Tool calls:', trace.tool_calls)
print(f'\n--- Final ({trace.steps} step(s), {trace.latency_s}s) ---')
print(trace.final_answer)

Tool calls: []

--- Final (2 step(s), 15.149s) ---
The calculated anion gap is 46.7 mmHg, which falls within the normal range of 70-100 mmHg in adults. This indicates that there may be some hemodynamic compromise, but it is not severe enough to warrant a clinical diagnosis of hypertension or hyperkalemia. The lab results suggest that the patient's blood pressure is within normal ranges and does not require further evaluation.


## Path 3 — computation + retrieval combined

In [4]:
trace = run_pipeline_react(
    'Patient height 1.68 m, weight 82 kg. Compute BMI and suggest next-step considerations.',
    llm=llm, retriever=retriever, tools=tools,
)
print('Tool calls:', trace.tool_calls)
print(f'\n--- Final ({trace.steps} step(s), {trace.latency_s}s) ---')
print(trace.final_answer)

Tool calls: []

--- Final (4 step(s), 25.452s) ---
- Calculated Parameter: BMI (mmHg x kg^2) = 168 / 1.68 * 703 / 1000 = 59.4
- Hypothese: The patient's weight is within the normal range of 18.5-24.9, but their BMI is above 25. Suggest next-step consideration for monitoring appetite and activity level, consulting a licensed clinician if necessary.


## Inspecting raw model outputs

Every step's raw model output is preserved in `trace.raw_outputs`. Useful for understanding why a step branched the way it did.

In [5]:
for i, raw in enumerate(trace.raw_outputs, 1):
    print(f'=== STEP {i} RAW MODEL OUTPUT ===')
    print(raw)
    print()

=== STEP 1 RAW MODEL OUTPUT ===
Based on the patient's height of 1.68 meters and weight of 82 kilograms, the following BMI calculation is performed:

BMI = (weight in kg) / (height in m)^2

The calculated BMI is 24.9, which falls within the normal range for adults between 18.5-24.9 kg/m^2. The next-step consideration suggested by the tool is to monitor the patient's weight and height regularly to ensure that they are maintaining a healthy BMI.

Additionally, based on the patient's age of 30 years, the following hypotheses and next-step considerations are generated:

Hypothesis: The patient may have a family history of obesity or diabetes.

Next-Step Consideration: The patient should monitor their weight and height regularly to ensure that they maintain a healthy BMI. If the patient's weight increases above 25 kg/m^2, they should seek medical attention.

Hypothesis: The patient may have a sedentary lifestyle or poor dietary habits.

Next-Step Consideration: The patient should prioritize